# Using Machine Learning to Solve for RSI SMA period
Selection system: two approaches
- Create an algo that performs sentiment and fundamental analysis on crypto, giving it a score then we pick the best performers
- Identify by price action. Price movement 15m, 1hr, 4hr all positive. Then volume.

In [1]:
# Add the project root (one level up from /research) to the path
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
from utils.indicators import *
from utils.engine import *

df_orderbook = pd.read_csv("./csv/BTCUSDT_orderbook.csv")
df_trades = pd.read_csv("./csv/BTCUSDT_trades.csv")
# Convert unix timestamp to datetime
df_orderbook['datetime'] = pd.to_datetime(df_orderbook['timestamp'], unit='ms')  # change unit to 's' if in seconds
df_trades['datetime'] = pd.to_datetime(df_trades['timestamp'], unit='ms')

# Set datetime as index
df_orderbook = df_orderbook.set_index('datetime')
df_trades = df_trades.set_index('datetime')

# Aggregate orderbook to 1m (mid-price OHLCV style)
df_orderbook['mid_price'] = (df_orderbook['bid_price'] + df_orderbook['ask_price']) / 2
df_ob_1m = df_orderbook.resample('1min').agg({
    'mid_price': ['first', 'max', 'min', 'last'],
    'bid_price': 'last',
    'ask_price': 'last',
    'bid_amount': 'sum',
    'ask_amount': 'sum',
})
df_ob_1m.columns = ['open', 'high', 'low', 'close', 'bid_price', 'ask_price', 'bid_amount', 'ask_amount']

df_ob_1m['mid_price'] = (df_ob_1m['bid_price'] + df_ob_1m['ask_price']) / 2
df_ob_1m['spread'] = df_ob_1m['ask_price'] - df_ob_1m['bid_price']

df_trades['buy_vol']  = df_trades['amount'].where(df_trades['side'] == 'buy', 0)
df_trades['sell_vol'] = df_trades['amount'].where(df_trades['side'] == 'sell', 0)

# Aggregate trades to 1m
df_trades_1m = df_trades.resample('1min').agg({
    'price': ['first', 'max', 'min', 'last', 'count'],
    'amount': 'sum',
    'buy_vol': ['sum', 'max'],
    'sell_vol': ['sum', 'max']
})

# Flatten multi-level columns
df_trades_1m.columns = ['open', 'high', 'low', 'close', 'num_trades', 'volume', 'agg_buyer', 'largest_buyer', 'agg_seller', 'largest_seller']

df_trades_1m = df_trades_1m.dropna(subset=['close'])

df_price_1m = pd.DataFrame({
    'open': df_ob_1m['open'],
    'high': df_ob_1m['high'],
    'low': df_ob_1m['low'],
    'close': df_ob_1m['close'],
    'largest_buyer': df_trades_1m['largest_buyer'] * 100 / df_trades_1m['volume'],
    'largest_seller': df_trades_1m['largest_seller'] * 100 / df_trades_1m['volume'],
    'OFI_LOB': 100*(df_ob_1m['bid_amount'] - df_ob_1m['ask_amount']) / (df_ob_1m['bid_amount'] + df_ob_1m['ask_amount']),
    'OFI_AGG': 100*(df_trades_1m['agg_buyer'] - df_trades_1m['agg_seller']) / (df_trades_1m['agg_buyer'] + df_trades_1m['agg_seller']),
    'vol_ratio': volatility_ratio(df_ob_1m['high'], df_ob_1m['low'], df_ob_1m['close']),
    'candle_type': abs(df_ob_1m['close'] - df_ob_1m['open']) / abs(df_ob_1m['high'] - df_ob_1m['low']),
    'candle_size': (df_ob_1m['high'] - df_ob_1m['low']) / (df_ob_1m['high'] - df_ob_1m['low']).rolling(14).mean(),
})
df_price_1m

,open,high,low,close,largest_buyer,largest_seller,OFI_LOB,OFI_AGG,vol_ratio,candle_type,candle_size
datetime,,,,,,,,,,,
2026-06-28 08:59:00,60343.75,60343.75,60337.35,60337.35,14.248252,23.688811,-74.522379,-39.219114,NaN,1.000000,NaN
2026-06-28 09:00:00,60337.35,60353.45,60334.95,60344.15,7.280285,7.939430,-26.065918,-0.142518,NaN,0.367568,NaN
2026-06-28 09:01:00,60344.15,60348.75,60311.65,60318.35,4.612850,5.826469,-18.319465,-59.648545,NaN,0.695418,NaN
2026-06-28 09:02:00,60318.35,60338.15,60318.35,60338.15,7.604817,5.352364,43.894900,8.206958,NaN,1.000000,NaN
2026-06-28 09:03:00,60338.15,60338.15,60276.75,60286.25,3.312667,3.344345,-19.263835,-13.743947,NaN,0.845277,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2026-06-28 13:26:00,60098.45,60098.45,60066.25,60075.55,5.419730,2.142816,-1.054677,4.188750,0.904207,0.711180,1.004680
2026-06-28 13:27:00,60075.55,60086.65,60054.95,60071.95,3.404592,5.911850,16.933574,-5.252045,0.907620,0.113565,1.009554
2026-06-28 13:28:00,60071.95,60092.15,60071.95,60089.55,2.386321,9.526494,-13.225270,-28.786171,0.849512,0.871287,0.654175


# Extracting Target Variable and Advanced Data Manipulation

In [2]:
price_clean = np.asanyarray(df_price_1m['close'], dtype=float)
rsi_2 = get_wilders_rsi(np.diff(price_clean), 2)
rsi_14 = get_wilders_rsi(np.diff(price_clean), 14)
rsi_14_sma = sma(rsi_14, 14)

# 1. Calculate the indicators
price_clean = np.asanyarray(df_price_1m['close'], dtype=float)
diffs = np.diff(price_clean)

# 2. Get the raw RSI arrays
rsi_14_raw = get_wilders_rsi(diffs, 14)
rsi_14_sma_raw = sma(rsi_14_raw, 14)

# 3. Create full-length arrays filled with NaN
rsi_14 = np.full(len(df_price_1m), np.nan)
rsi_14_sma = np.full(len(df_price_1m), np.nan)

rsi_14[-len(rsi_14_raw):] = rsi_14_raw
rsi_14_sma[-len(rsi_14_sma_raw):] = rsi_14_sma_raw

# 5. Assign to DataFrame
df_price_1m['rsi_14'] = rsi_14
df_price_1m['sma_14'] = rsi_14_sma

# Call it with your variables, explicitly passing labels so the chart is clear
df_price_1m['rsi_above_sma'] = df_price_1m['rsi_14'] > df_price_1m['sma_14']

# 2. Identify where the condition changes (this creates a new "block" for every switch)
df_price_1m['block'] = (df_price_1m['rsi_above_sma'] != df_price_1m['rsi_above_sma'].shift()).cumsum()

# 3. Group by the condition and block to count the duration of each streak
streak_durations = df_price_1m.groupby(['rsi_above_sma', 'block']).size().reset_index(name='duration')

import numpy as np
import pandas as pd
from sklearn.ensemble           import GradientBoostingClassifier
from sklearn.preprocessing      import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection    import TimeSeriesSplit
from sklearn.metrics            import classification_report, confusion_matrix, f1_score
import warnings
warnings.filterwarnings('ignore')

df = df_price_1m.reset_index(drop=True).copy()
# ─────────────────────────────────────────────────────────────────────────────
# 0 · Setup
# ─────────────────────────────────────────────────────────────────────────────

# ── Recompute block in case the df was rebuilt or the column is stale ────────
df['rsi_above_sma'] = df['rsi_14'] > df['sma_14']
df['block'] = (df['rsi_above_sma'] != df['rsi_above_sma'].shift()).cumsum()

# ── Use transform instead of join — no KeyError, no index alignment issues ──
df['streak_dur'] = df.groupby('block')['block'].transform('count')
is_start        = df['block'] != df['block'].shift()
start_positions = df.index[is_start].tolist()

CLASS_NAMES = ['Short (≤3)', 'Medium (4–10)', 'Long (>10)']

def label(d):
    return 0 if d <= 3 else (1 if d <= 10 else 2)

# ─────────────────────────────────────────────────────────────────────────────
# 1 · Feature extraction — at bar-0 of each new streak only
# ─────────────────────────────────────────────────────────────────────────────

LOOKBACK = 10          # bars of history needed before the crossover

records = []

for i, pos in enumerate(start_positions):
    if pos < LOOKBACK:
        continue

    row  = df.loc[pos]
    hist = df.iloc[pos - LOOKBACK : pos]   # the 10 bars BEFORE crossover (no leakage)
    prev = start_positions[i - 1] if i > 0 else None

    # ── RSI / SMA ──────────────────────────────────────────────────────────
    rsi     = row['rsi_14']
    sma_val = row['sma_14']       # ← safe name, no collision

    # ── Order-flow at the crossover bar ────────────────────────────────────
    ofi_lob = row['OFI_LOB']
    ofi_agg = row['OFI_AGG']

    # ── Order-flow in the run-up (last 5 and 10 bars) ──────────────────────
    ofi_lob_5  = hist.tail(5)['OFI_LOB'].mean()
    ofi_agg_5  = hist.tail(5)['OFI_AGG'].mean()
    ofi_lob_10 = hist['OFI_LOB'].mean()
    ofi_agg_10 = hist['OFI_AGG'].mean()

    # Cumulative order-flow pressure building into the crossover
    ofi_lob_cum = hist['OFI_LOB'].sum()
    ofi_agg_cum = hist['OFI_AGG'].sum()

    # ── Volume imbalance ───────────────────────────────────────────────────
    vr      = row['vol_ratio']
    vr_5    = hist.tail(5)['vol_ratio'].mean()
    vr_10   = hist['vol_ratio'].mean()

    # ── Buyer / seller pressure ────────────────────────────────────────────
    lb       = row['largest_buyer']
    ls       = row['largest_seller']
    bs_ratio = lb / (ls + 1e-9)           # >1 → buyer dominant

    lb_5_mean = hist.tail(5)['largest_buyer'].mean()
    ls_5_mean = hist.tail(5)['largest_seller'].mean()

    # ── Candle character at crossover ──────────────────────────────────────
    c_type = row['candle_type']
    c_size = row['candle_size']

    c_size_5_mean = hist.tail(5)['candle_size'].mean()
    c_type_5_mean = hist.tail(5)['candle_type'].mean()   # avg sentiment

    # ── RSI dynamics ───────────────────────────────────────────────────────
    rsi_sma_dist = rsi - sma_val
    rsi_vs_50    = rsi - 50
    rsi_d3       = rsi - df.at[pos - 3, 'rsi_14']
    rsi_d5       = rsi - df.at[pos - 5, 'rsi_14']

    rec = {
        # RSI / SMA
        'rsi':            rsi,
        'rsi_sma_dist':   rsi_sma_dist,
        'rsi_vs_50':      rsi - 50,
        'rsi_d3':         rsi_d3,
        'rsi_d5':         rsi_d5,
        'direction':      int(row['rsi_above_sma']),   # 1=above, 0=below

        # Order flow — spot
        'ofi_lob':        ofi_lob,
        'ofi_agg':        ofi_agg,
        'ofi_alignment':  int(np.sign(ofi_lob) == np.sign(ofi_agg)),  # both agree?

        # Order flow — run-up
        'ofi_lob_5':      ofi_lob_5,
        'ofi_agg_5':      ofi_agg_5,
        'ofi_lob_10':     ofi_lob_10,
        'ofi_agg_10':     ofi_agg_10,
        'ofi_lob_cum':    ofi_lob_cum,
        'ofi_agg_cum':    ofi_agg_cum,

        # Volume
        'vol_ratio':      vr,
        'vol_ratio_5':    vr_5,
        'vol_ratio_10':   vr_10,

        # Buyer / seller
        'largest_buyer':  lb,
        'largest_seller': ls,
        'bs_ratio':       bs_ratio,
        'lb_5_mean':      lb_5_mean,
        'ls_5_mean':      ls_5_mean,

        # Candle
        'candle_type':    c_type,
        'candle_size':    c_size,
        'candle_size_5':  c_size_5_mean,
        'candle_type_5':  c_type_5_mean,

        # Streak memory (previous streak is already complete — no leakage)
        'prev_dur':       df.at[prev, 'streak_dur'] if prev is not None else np.nan,

        'target':         label(row['streak_dur']),
        'streak_dur':     row['streak_dur'],
    }

    records.append(rec)

feat_df = pd.DataFrame(records).dropna()

print(f"Usable streaks: {len(feat_df)}")
print("\nClass distribution:")
print(feat_df['target'].value_counts().sort_index()
      .rename(index=dict(enumerate(CLASS_NAMES))))

# ─────────────────────────────────────────────────────────────────────────────
# 2 · Train / test split  (chronological — never shuffle)
# ─────────────────────────────────────────────────────────────────────────────

META  = ['target', 'streak_dur']
FEATS = [c for c in feat_df.columns if c not in META]

X = feat_df[FEATS].values
y = feat_df['target'].values

cut        = int(len(feat_df) * 0.80)
X_tr, X_te = X[:cut], X[cut:]
y_tr, y_te = y[:cut], y[cut:]

scaler  = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

sw_tr = compute_sample_weight('balanced', y_tr)

# ─────────────────────────────────────────────────────────────────────────────
# 3 · Train
# ─────────────────────────────────────────────────────────────────────────────

clf = GradientBoostingClassifier(
    n_estimators     = 300,
    max_depth        = 3,
    learning_rate    = 0.05,
    subsample        = 0.8,
    min_samples_leaf = 5,
    random_state     = 42,
)
clf.fit(X_tr_s, y_tr, sample_weight=sw_tr)

# ─────────────────────────────────────────────────────────────────────────────
# 4 · Evaluate
# ─────────────────────────────────────────────────────────────────────────────
present_classes = sorted(np.unique(np.concatenate([y_tr, y_te])))
present_names   = [CLASS_NAMES[c] for c in present_classes]

y_pred = clf.predict(X_te_s)
proba  = clf.predict_proba(X_te_s)

print("\n── Classification Report ─────────────────────────────────────────")
print(classification_report(y_te, y_pred,
                             labels      = present_classes,
                             target_names= present_names))

cm = confusion_matrix(y_te, y_pred, labels=present_classes)
print("── Confusion Matrix ──────────────────────────────────────────────")
print(pd.DataFrame(cm,
      index   = [f"True  {n}" for n in present_names],
      columns = [f"Pred  {n}" for n in present_names]))

tscv  = TimeSeriesSplit(n_splits=5)
cv_f1 = []
for tr_idx, val_idx in tscv.split(X_tr_s):
    sw_f = compute_sample_weight('balanced', y_tr[tr_idx])
    m = GradientBoostingClassifier(n_estimators=300, max_depth=3,
            learning_rate=0.05, subsample=0.8, min_samples_leaf=5, random_state=42)
    m.fit(X_tr_s[tr_idx], y_tr[tr_idx], sample_weight=sw_f)
    cv_f1.append(f1_score(y_tr[val_idx], m.predict(X_tr_s[val_idx]),
                          labels=present_classes, average='macro'))

print(f"\nWalk-forward macro-F1: {np.mean(cv_f1):.3f} ± {np.std(cv_f1):.3f}")

Usable streaks: 33

Class distribution:
target
Short (≤3)       13
Medium (4–10)     6
Long (>10)       14
Name: count, dtype: int64

── Classification Report ─────────────────────────────────────────
               precision    recall  f1-score   support

   Short (≤3)       0.50      0.50      0.50         2
Medium (4–10)       0.00      0.00      0.00         0
   Long (>10)       0.80      0.80      0.80         5

     accuracy                           0.71         7
    macro avg       0.43      0.43      0.43         7
 weighted avg       0.71      0.71      0.71         7

── Confusion Matrix ──────────────────────────────────────────────
                     Pred  Short (≤3)  Pred  Medium (4–10)  Pred  Long (>10)
True  Short (≤3)                    1                    0                 1
True  Medium (4–10)                 0                    0                 0
True  Long (>10)                    1                    0                 4

Walk-forward macro-F1: 0.293 ± 0.19

# List of Features We Want to Train
